In [2]:
import os
import json
import shutil
import glob
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# ==========================================
# [사용자 설정 구간]
# ==========================================
RAW_DATA_ROOT = r"C:\Users\user\Desktop\data"  
OUTPUT_DATASET_ROOT = r"C:\Users\user\Desktop\SchoolZone_YOLO_Dataset"

# 💡 수정 포인트 1: 소분류를 무시하고 대분류(Super Class) 기준으로만 넓게 잡습니다.
CLASS_MAPPING = {
    'person': 0,      # 사람(보행자, 어린이 등 모두 포함)
    'vehicle': 1,     # 차량(승용차, 버스, 트럭 등 모두 포함)
    'road_etc': 2     # 시설물(횡단보도, 방지턱 등 모두 포함)
}
CLASS_NAMES = ['person', 'vehicle', 'road_etc']

IMAGE_SIZE = (1920, 1080)
# ==========================================

def convert_bbox_to_yolo(size, box):
    dw, dh = 1. / size[0], 1. / size[1]
    x = (box[0] + box[2] / 2.0) * dw
    y = (box[1] + box[3] / 2.0) * dh
    w = box[2] * dw
    h = box[3] * dh
    return (x, y, w, h)

# 1. 폴더 구조 생성
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(OUTPUT_DATASET_ROOT, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DATASET_ROOT, split, 'labels'), exist_ok=True)

# 2. 어노테이션 파일 수집
print("🔍 데이터를 탐색하는 중...")
annotation_path_pattern = os.path.join(RAW_DATA_ROOT, "**", "1.어노테이션", "**", "*.json")
all_json_files = glob.glob(annotation_path_pattern, recursive=True)
print(f"✅ 총 {len(all_json_files)}개의 어노테이션 파일을 발견했습니다.")

# 3. 데이터 세트 분할
train_files, test_files = train_test_split(all_json_files, test_size=0.2, random_state=42)
val_files, test_files = train_test_split(test_files, test_size=0.5, random_state=42)

def process_data_split(json_files, split_name):
    print(f"📦 {split_name} 데이터셋 구축 시작...")
    count = 0
    missing_image_count = 0
    
    for json_path in tqdm(json_files):
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            yolo_labels = []
            
            # 어노테이션 파싱
            for ann in data.get('annotations', []):
                super_cls = ann.get('object super class')  # 대분류만 확인
                
                if super_cls in CLASS_MAPPING:
                    cls_id = CLASS_MAPPING[super_cls]
                    bbox = ann.get('bbox')
                    if bbox:
                        yolo_box = convert_bbox_to_yolo(IMAGE_SIZE, bbox)
                        yolo_labels.append(f"{cls_id} {' '.join(map(str, yolo_box))}")
            
            if yolo_labels:
                file_stem = Path(json_path).stem
                
                # 💡 수정 포인트 2: 경로에서 VL을 VS로 변경하여 이미지 찾기
                # 예: ...\VL\1.어노테이션\... -> ...\VS\1.어노테이션\...
                img_src_path = str(json_path).replace("\\VL\\", "\\VS\\").replace(".json", ".jpg")
                
                if os.path.exists(img_src_path):
                    # 라벨 저장
                    label_out_path = os.path.join(OUTPUT_DATASET_ROOT, split_name, 'labels', f"{file_stem}.txt")
                    with open(label_out_path, 'w') as f:
                        f.write("\n".join(yolo_labels))
                    
                    # 이미지 복사
                    img_out_path = os.path.join(OUTPUT_DATASET_ROOT, split_name, 'images', f"{file_stem}.jpg")
                    shutil.copy(img_src_path, img_out_path)
                    count += 1
                else:
                    missing_image_count += 1

        except Exception as e:
            pass # 에러 무시하고 계속 진행

    print(f"✅ {split_name} 완료: {count}개의 세트 정리 성공! (이미지 없음: {missing_image_count}개)")

process_data_split(train_files, 'train')
process_data_split(val_files, 'val')
process_data_split(test_files, 'test')

# 4. data.yaml 파일 생성
yaml_content = f"""path: {OUTPUT_DATASET_ROOT}
train: train/images
val: val/images
test: test/images

nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
"""
with open(os.path.join(OUTPUT_DATASET_ROOT, 'data.yaml'), 'w') as f:
    f.write(yaml_content)

print("\n🎉 모든 작업 완료!")

🔍 데이터를 탐색하는 중...
✅ 총 84297개의 어노테이션 파일을 발견했습니다.
📦 train 데이터셋 구축 시작...


100%|██████████| 67437/67437 [40:04<00:00, 28.05it/s]


✅ train 완료: 67162개의 세트 정리 성공! (이미지 없음: 167개)
📦 val 데이터셋 구축 시작...


100%|██████████| 8430/8430 [05:53<00:00, 23.86it/s]


✅ val 완료: 8392개의 세트 정리 성공! (이미지 없음: 21개)
📦 test 데이터셋 구축 시작...


100%|██████████| 8430/8430 [05:41<00:00, 24.65it/s]

✅ test 완료: 8392개의 세트 정리 성공! (이미지 없음: 23개)

🎉 모든 작업 완료!


In [4]:
from ultralytics import YOLO

# 1. 모델 로드: 노트북 사양을 고려해 가장 가볍고 빠른 Nano 모델을 선택합니다.
model = YOLO('yolo11n.pt')  # 또는 'yolov8n.pt'

# 2. 테스트 학습 시작 (Sanity Check)
results = model.train(
    data=r'C:\Users\user\Desktop\SchoolZone_YOLO_Dataset\data.yaml',
    epochs=3,        # 💡 에러 없이 끝까지 도는지 확인하기 위해 3번만 돌려봅니다.
    imgsz=640,       # 기본 이미지 사이즈
    batch=8,         # 💡 학습 중 'Out of Memory' 에러가 나면 4나 2로 줄여주세요.
    device='cpu',        # GPU 사용 (GPU 인식이 안 되어 에러가 나면 'cpu'로 변경)
    workers=2,       # 데이터 로딩 속도 조절 (노트북 과부하 방지)
    project='SchoolZone_Project',
    name='test_run_01'
)

New https://pypi.org/project/ultralytics/8.4.30 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.27  Python-3.10.19 torch-2.11.0+cpu CPU (Intel Core Ultra 7 155H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\user\Desktop\SchoolZone_YOLO_Dataset\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.

KeyboardInterrupt: 